In [35]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, beta

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

COLS = {
    "CFG_algorythm":                    "pattern",
    "CFG_node_distance":                "A",        # wire segment length, mm
    "CFG_joint_radius":                 "J",        # jumper length, mm
    "CFG_clothe_ID":                    "clothing_id",
    "CFG_size":                         "size",
    "SP_P_reachable_sensors":           "reach",    # shortest-path sensor reachability
    "SP_P_unreachable_bc_short_jumper": "unreach_geom",
    "T_wire_L":                         "wire",     # total wire length, mm
    "T_jumper_C":                       "jumpers",
    "T_node_C":                         "nodes",
    "SP_path_L__Mean":                  "path_mean",
    "SP_path_L__Max":                   "path_max",
    "SP_path_jumper_C__Mean":           "path_jumpers",
}

def load(path, run, tag=""):
    df = pd.read_csv(path, low_memory=False)
    missing = [c for c in COLS if c not in df.columns]
    if missing:
        raise KeyError("missing columns in %s: %s" % (path, missing))
    out = df[list(COLS)].rename(columns=COLS)
    out["run"] = df["CFG_run"] if "CFG_run" in df.columns else run
    out["tag"] = df["CFG_tag"] if "CFG_tag" in df.columns else tag
    return out

a = load("../results/integrated_results.csv", "initial_2023")
b = load("../results_ext/integrated_results_ext.csv", "extended_2026")
d = pd.concat([a, b], ignore_index=True)

# wire in metres; pattern efficiency as defined in section 4.6
d["wire_m"] = d.wire / 1000.0
d["EP"] = d.reach * 100 / d.wire_m * 1000

FINALISTS = ["3.3.3.3.3.3", "4.4.4.4", "3.6.3.6"]
SPACING = {"3.3.3.3.3.3": 0.8660254, "4.4.4.4": 1.0, "3.6.3.6": 1.7320508}
d["D"] = d.A * d.pattern.map(SPACING)      # NaN for non-finalists, by design

print("initial  :", len(a))
print("extended :", len(b))
print("combined :", len(d))
print()
print(d.groupby(["run", "tag"]).size().to_string())
print()
cells = d.groupby(["pattern", "A", "J"]).size()
print("configuration cells:", len(cells), " incomplete:", dict(cells[cells < 170]))

initial  : 44200
extended : 41988
combined : 86188

run            tag   
extended_2026  tier1      6120
               tier2     10200
               tier3      6120
               tier4     10200
               tier5      3740
               tier5b     1358
               tier6      1190
               tier7      3060
initial_2023             44200

configuration cells: 504  incomplete: {('3.6.3.6', 13.0, 10.0): np.int64(168)}


In [36]:
def cp_lower(k, n, alpha=0.05):
    """One-sided Clopper-Pearson lower bound on a proportion."""
    return 0.0 if k == 0 else beta.ppf(alpha, k, n - k + 1)


def design_means(sub):
    """Mean reachability per garment design (the independent unit, n=17)."""
    return sub.groupby("clothing_id").reach.mean()


def variant_pass(sub, r_min):
    """Pass/fail per garment variant (design x size), with design labels for clustering."""
    v = sub.groupby(["clothing_id", "size"]).reach.mean().reset_index()
    v["pass"] = v.reach >= r_min
    return v


def cluster_lb(v, reps=2000, seed=0):
    """One-sided 95% lower bound on the variant pass proportion,
    bootstrapping whole designs so the nesting of sizes is respected."""
    rng = np.random.default_rng(seed)
    by = {k: g["pass"].values for k, g in v.groupby("clothing_id")}
    keys = list(by)
    draws = [np.concatenate([by[k] for k in rng.choice(keys, len(keys), replace=True)]).mean()
             for _ in range(reps)]
    return np.percentile(draws, 5)


def build_cells(d, r_min=0.95, reps=2000):
    rows = []
    for (pattern, A, J), sub in d.groupby(["pattern", "A", "J"]):
        per = design_means(sub)
        if len(per) < 17:
            continue                      # excludes the one incomplete cell
        v = variant_pass(sub, r_min)
        k = int((per >= r_min).sum())
        rows.append(dict(
            pattern=pattern, A=A, J=J,
            D=A * SPACING.get(pattern, np.nan),
            R_mean=per.mean(), R_median=per.median(), R_min_design=per.min(),
            wire_m=sub.wire_m.mean(), EP=sub.EP.mean(),
            jumpers=sub.jumpers.mean(), nodes=sub.nodes.mean(),
            unreach_geom=sub.unreach_geom.mean(),
            designs_ok=k,
            p_wilcoxon=(wilcoxon(per - r_min, alternative="greater").pvalue
                        if per.std() > 1e-12 else (0.0 if per.mean() > r_min else 1.0)),
            variant_pt=v["pass"].mean(),
            variant_lb=cluster_lb(v, reps=reps),
            design_lb=cp_lower(k, 17),
        ))
    return pd.DataFrame(rows)


cells95 = build_cells(d, r_min=0.95)
print(len(cells95), "complete configuration cells")
cells95.head()

504 complete configuration cells


,pattern,A,J,D,R_mean,R_median,R_min_design,wire_m,EP,jumpers,nodes,unreach_geom,designs_ok,p_wilcoxon,variant_pt,variant_lb,design_lb
0,3.12.12,20.0,10.0,NaN,0.496341,0.500177,0.470994,44.726260,1513.368143,34.929412,1692.282353,0.499438,0,1.000000,0.000000,0.000000,0.000000
1,3.12.12,20.0,20.0,NaN,0.795820,0.796275,0.793919,44.740418,2425.139092,68.741176,1692.705882,0.204180,0,1.000000,0.000000,0.000000,0.000000
2,3.12.12,20.0,40.0,NaN,0.999731,0.999758,0.999510,44.710621,3054.718345,131.064706,1690.647059,0.000269,17,0.000008,1.000000,1.000000,0.838434
3,3.12.12,20.0,80.0,NaN,0.995384,1.000000,0.963408,44.644647,3044.181743,246.052941,1689.317647,0.003675,17,0.000066,0.988235,0.976471,0.838434
4,3.12.12,20.0,160.0,NaN,1.000000,1.000000,1.000000,44.748383,3048.603378,428.570588,1693.435294,0.000000,17,0.000000,1.000000,1.000000,0.838434


In [37]:
def selection_table(cells, patterns=None, criterion="variant_lb", q=0.90):
    sel = cells if patterns is None else cells[cells.pattern.isin(patterns)]
    rows = []
    for J in sorted(sel.J.unique()):
        s = sel[sel.J == J]
        s = s[s.variant_lb >= q] if criterion == "variant_lb" else s[s.p_wilcoxon < 0.05]
        if not len(s):
            continue
        b = s.loc[s.EP.idxmax()]
        rows.append(dict(J=int(J), pattern=b.pattern,
                         D=(round(b.D) if pd.notna(b.D) else None), A=int(b.A),
                         R_mean=round(b.R_mean * 100, 2),
                         worst=round(b.R_min_design * 100, 2),
                         wire_m=round(b.wire_m, 2), EP=round(b.EP),
                         designs=int(b.designs_ok),
                         var_pt=round(b.variant_pt * 100, 1),
                         var_lb=round(b.variant_lb * 100, 1),
                         p=round(b.p_wilcoxon, 4)))
    return pd.DataFrame(rows)


for q in [0.70, 0.80, 0.90, 0.95]:
    t = selection_table(cells95, patterns=FINALISTS, q=q)
    print("q >= %.0f%%" % (q * 100))
    print(t.to_string(index=False))
    print()

print("Wilcoxon criterion (design-level test, p<0.05):")
print(selection_table(cells95, patterns=FINALISTS, criterion="p_wilcoxon").to_string(index=False))



q >= 70%
  J pattern   D   A  R_mean  worst  wire_m    EP  designs  var_pt  var_lb      p
 10 4.4.4.4  24  24   97.04  97.00   99.92  1325       17   100.0   100.0 0.0000
 20 4.4.4.4  50  50   95.53  95.37   47.91  2722       17    98.2    96.5 0.0000
 40 4.4.4.4  96  96   96.18  95.55   24.88  5280       17    96.5    93.5 0.0000
 80 4.4.4.4 180 180   94.85  87.38   13.17  9729       10    80.0    72.9 0.4088
160 4.4.4.4 249 249   92.94  83.95    9.44 13139        9    77.6    72.9 0.8111

q >= 80%
  J pattern   D   A  R_mean  worst  wire_m    EP  designs  var_pt  var_lb      p
 10 4.4.4.4  24  24   97.04  97.00   99.92  1325       17   100.0   100.0 0.0000
 20 4.4.4.4  50  50   95.53  95.37   47.91  2722       17    98.2    96.5 0.0000
 40 4.4.4.4  96  96   96.18  95.55   24.88  5280       17    96.5    93.5 0.0000
 80 4.4.4.4 170 170   97.44  93.48   13.98  9435       16    97.1    95.3 0.0001
160 4.4.4.4 243 243   95.16  89.93    9.73 12956        9    86.5    82.9 0.3910

q >= 90%

In [38]:
def coverage_curve(cells, pattern, J):
    s = cells[(cells.pattern == pattern) & (cells.J == J)].sort_values("D")
    return s.D.values, s.variant_lb.values


def D_at_q(cells, pattern, J, q):
    """Largest spacing whose coverage bound reaches q, linearly interpolated.
    Returns (D, bracket_width_pct) - a wide bracket means the grid is coarse there."""
    D, lb = coverage_curve(cells, pattern, J)
    idx = np.where(lb >= q)[0]
    if not len(idx):
        return np.nan, np.nan
    i = idx[-1]
    if i + 1 >= len(D) or lb[i] <= q:
        return D[i], 0.0
    frac = (lb[i] - q) / (lb[i] - lb[i + 1])
    return D[i] + frac * (D[i + 1] - D[i]), (D[i + 1] / D[i] - 1) * 100


def guideline_table(cells, pattern="4.4.4.4", qs=(0.70, 0.80, 0.90, 0.95, 0.99)):
    rows = []
    for J in sorted(cells[cells.pattern == pattern].J.unique()):
        row = {"J": int(J)}
        for q in qs:
            Dq, br = D_at_q(cells, pattern, J, q)
            row["q=%.0f%%" % (q * 100)] = ("%.0f%s" % (Dq, "*" if br > 15 else "")
                                           if np.isfinite(Dq) else "-")
        rows.append(row)
    return pd.DataFrame(rows)


print("WIRE SPACING D (mm), 4.4.4.4, R_min = 95%")
print("* = interpolated across a bracket wider than 15%, treat as indicative")
print(guideline_table(cells95).to_string(index=False))

print("\ncoverage curve at J=160 (shows the gap):")
D, lb = coverage_curve(cells95, "4.4.4.4", 160)
print(pd.DataFrame({"D": D.astype(int), "coverage_lb_%": (lb * 100).round(1)}).to_string(index=False))



WIRE SPACING D (mm), 4.4.4.4, R_min = 95%
* = interpolated across a bracket wider than 15%, treat as indicative
  J q=70% q=80% q=90% q=95% q=99%
 10   25*   25*   24*   24*   24*
 20    51    51    50    50   43*
 40    98    97    96    94    86
 80   181   174   171   170  134*
160  256*   244   194   168  145*

coverage curve at J=160 (shows the gap):
  D  coverage_lb_%
 20          100.0
 40          100.0
 80          100.0
125          100.0
160           98.2
170           94.1
180           94.1
190           92.4
198           87.6
206           88.2
224           86.5
240           74.1
243           82.9
246           72.9
249           72.9
320           44.7
357           24.1
412            0.6
480            0.0
640            0.0
714            0.0


In [39]:
from scipy.interpolate import PchipInterpolator

def critical_spacing(d, pattern, J, r_min=0.95):
    """D* per garment variant: the spacing at which its reachability crosses r_min.
    Monotone (PCHIP) interpolation of the per-variant reachability curve."""
    s = d[(d.pattern == pattern) & (d.J == J)]
    out = {}
    for (cid, size), g in s.groupby(["clothing_id", "size"]):
        g = g.groupby("A").reach.mean().sort_index()
        if len(g) < 3:
            continue
        D = g.index.values * SPACING[pattern]
        r = g.values
        if r[0] < r_min:                       # fails even at the finest spacing
            out[(cid, size)] = 0.0
            continue
        if r[-1] >= r_min:                     # still passing at the coarsest
            out[(cid, size)] = np.inf
            continue
        # invert the monotone curve: find D where reach crosses r_min
        f = PchipInterpolator(D, r - r_min, extrapolate=False)
        grid = np.linspace(D[0], D[-1], 4000)
        v = f(grid)
        cross = np.where(np.diff(np.sign(v)) < 0)[0]
        out[(cid, size)] = grid[cross[0]] if len(cross) else D[-1]
    return pd.Series(out, name="D_star")


def coverage_at(D_star, D):
    return float((D_star > D).mean())


def D_at_q_exact(D_star, q, hi=800.0):
    """Largest spacing whose coverage is at least q. Exact given D*, no proportion interpolated."""
    lo = 0.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if coverage_at(D_star, mid) >= q:
            lo = mid
        else:
            hi = mid
    return lo


ds = critical_spacing(d, "4.4.4.4", 160)
print("J=160, 4.4.4.4 : D* across 170 garment variants")
print("  min %.0f  q05 %.0f  median %.0f  q95 %.0f  max %s"
      % (ds.min(), ds.quantile(.05), ds.median(), ds.quantile(.95),
         "inf" if np.isinf(ds.max()) else "%.0f" % ds.max()))
print()
for q in [0.70, 0.80, 0.90, 0.95, 0.99]:
    print("  q=%.0f%%  ->  D = %.0f mm" % (q * 100, D_at_q_exact(ds, q)))



J=160, 4.4.4.4 : D* across 170 garment variants
  min 135  q05 182  median 261  q95 376  max 418

  q=70%  ->  D = 240 mm
  q=80%  ->  D = 228 mm
  q=90%  ->  D = 192 mm
  q=95%  ->  D = 182 mm
  q=99%  ->  D = 162 mm


In [40]:
def D_at_q_boot(D_star, q, reps=2000, seed=0, hi=800.0):
    """Cluster-bootstrap lower bound on the spacing achieving coverage q.
    Resamples whole designs, carrying all their sizes."""
    rng = np.random.default_rng(seed)
    by = {}
    for (cid, size), v in D_star.items():
        by.setdefault(cid, []).append(v)
    keys = list(by)
    draws = []
    for _ in range(reps):
        pick = np.concatenate([by[k] for k in rng.choice(keys, len(keys), replace=True)])
        lo, h = 0.0, hi
        for _ in range(40):
            mid = (lo + h) / 2
            if (pick > mid).mean() >= q:
                lo = mid
            else:
                h = mid
        draws.append(lo)
    return np.percentile(draws, 5)


rows = []
for J in sorted(d[d.pattern == "4.4.4.4"].J.unique()):
    ds = critical_spacing(d, "4.4.4.4", J)
    row = {"J": int(J), "n": len(ds)}
    for q in [0.70, 0.80, 0.90, 0.95, 0.99]:
        row["q=%.0f%% pt" % (q * 100)] = round(D_at_q_exact(ds, q))
        row["q=%.0f%% lb" % (q * 100)] = round(D_at_q_boot(ds, q))
    rows.append(row)

guide = pd.DataFrame(rows)
print("WIRE SPACING D (mm), 4.4.4.4, R_min = 95%   (pt = point estimate, lb = 95% cluster-bootstrap lower bound)")
print(guide.to_string(index=False))



WIRE SPACING D (mm), 4.4.4.4, R_min = 95%   (pt = point estimate, lb = 95% cluster-bootstrap lower bound)
  J   n  q=70% pt  q=70% lb  q=80% pt  q=80% lb  q=90% pt  q=90% lb  q=95% pt  q=95% lb  q=99% pt  q=99% lb
 10 170        26        26        26        26        25        25        25        25        25        25
 20 170        51        51        51        51        50        50        50        50        50        49
 40 170        99        99        99        98        98        96        96        95        93        85
 80 170       176       173       171       170       166       165       161       161       134       131
160 170       240       230       228       201       192       182       182       164       162       135


In [41]:
ds = critical_spacing(d, "4.4.4.4", 40)
print("J=40 : D* distribution")
print("  min %.0f  q05 %.0f  q10 %.0f  median %.0f  max %s"
      % (ds.min(), ds.quantile(.05), ds.quantile(.10), ds.median(),
         "inf" if np.isinf(ds.max()) else "%.0f" % ds.max()))
print("  variants with D* below 100 mm:", int((ds < 100).sum()))
print()
print("simulated spacings at J=40:", sorted(d[(d.pattern=="4.4.4.4")&(d.J==40)].A.unique()))



J=40 : D* distribution
  min 85  q05 96  q10 97  median 100  max 104
  variants with D* below 100 mm: 82

simulated spacings at J=40: [np.float64(20.0), np.float64(40.0), np.float64(80.0), np.float64(90.0), np.float64(96.0), np.float64(100.0), np.float64(125.0), np.float64(160.0), np.float64(180.0), np.float64(206.0), np.float64(240.0), np.float64(320.0), np.float64(480.0), np.float64(640.0)]


In [42]:
def guideline_multi(d, pattern="4.4.4.4", r_mins=(0.90, 0.95, 0.99),
                    qs=(0.80, 0.90, 0.95)):
    out = {}
    for r in r_mins:
        rows = []
        for J in sorted(d[d.pattern == pattern].J.unique()):
            ds = critical_spacing(d, pattern, J, r_min=r)
            row = {"J": int(J)}
            for q in qs:
                row["q=%.0f%%" % (q * 100)] = round(D_at_q_boot(ds, q))
            rows.append(row)
        out[r] = pd.DataFrame(rows)
    return out


tables = guideline_multi(d)
for r, t in tables.items():
    print("R_min = %.0f%%   (wire spacing D in mm, 95%% cluster-bootstrap lower bound)" % (r * 100))
    print(t.to_string(index=False))
    print()



R_min = 90%   (wire spacing D in mm, 95% cluster-bootstrap lower bound)
  J  q=80%  q=90%  q=95%
 10     29     29     29
 20     57     57     57
 40    110    105    104
 80    171    166    162
160    210    193    173

R_min = 95%   (wire spacing D in mm, 95% cluster-bootstrap lower bound)
  J  q=80%  q=90%  q=95%
 10     26     25     25
 20     51     50     50
 40     98     96     95
 80    170    165    161
160    201    182    164

R_min = 99%   (wire spacing D in mm, 95% cluster-bootstrap lower bound)
  J  q=80%  q=90%  q=95%
 10     22     22     21
 20     44     43     43
 40     84     84     81
 80    139    126    106
160    164    137    117



In [43]:
plane = {"3.3.3.3.3.3": 4.4617 * 0.8660254, "4.4.4.4": 2.5760, "3.6.3.6": 1.5560 * 1.7320508}
rows = []
for p in FINALISTS:
    for J in sorted(d[d.pattern == p].J.unique()):
        ds = critical_spacing(d, p, J, r_min=0.95)
        if not len(ds):
            continue
        Dq = D_at_q_boot(ds, 0.90)
        rows.append(dict(pattern=p, J=int(J), D=round(Dq),
                         ratio=round(Dq / J, 2), plane=round(plane[p], 2),
                         frac=round(Dq / J / plane[p], 2)))
frontier = pd.DataFrame(rows)
print("FRONTIER: achieved D/J as a fraction of the infinite-plane prediction")
print("(R_min = 95%, q = 90%, cluster-bootstrap lower bound)")
print(frontier.pivot(index="J", columns="pattern", values="frac").to_string())
print()
print(frontier.to_string(index=False))



FRONTIER: achieved D/J as a fraction of the infinite-plane prediction
(R_min = 95%, q = 90%, cluster-bootstrap lower bound)
pattern  3.3.3.3.3.3  3.6.3.6  4.4.4.4
J                                     
10              0.99     0.99     0.99
20              0.96     0.99     0.98
40              0.86     0.95     0.93
80              0.68     0.84     0.80
160             0.37     0.54     0.44

    pattern   J   D  ratio  plane  frac
3.3.3.3.3.3  10  38   3.81   3.86  0.99
3.3.3.3.3.3  20  74   3.70   3.86  0.96
3.3.3.3.3.3  40 133   3.34   3.86  0.86
3.3.3.3.3.3  80 209   2.62   3.86  0.68
3.3.3.3.3.3 160 231   1.45   3.86  0.37
    4.4.4.4  10  25   2.54   2.58  0.99
    4.4.4.4  20  50   2.51   2.58  0.98
    4.4.4.4  40  96   2.41   2.58  0.93
    4.4.4.4  80 165   2.07   2.58  0.80
    4.4.4.4 160 182   1.14   2.58  0.44
    3.6.3.6  10  27   2.66   2.70  0.99
    3.6.3.6  20  53   2.66   2.70  0.99
    3.6.3.6  40 103   2.57   2.70  0.95
    3.6.3.6  80 181   2.26   2.70  0.84
  

In [44]:
def holm(ps):
    idx = np.argsort(ps); m = len(ps); adj = [0.0] * m; run = 0.0
    for k, i in enumerate(idx):
        v = max(run, (m - k) * ps[i]); run = v; adj[i] = min(v, 1.0)
    return adj


rec = []
for J in [10, 20, 40, 80, 160]:
    ds = critical_spacing(d, "4.4.4.4", J, r_min=0.95)
    Dq = D_at_q_boot(ds, 0.90)
    c = cells95[(cells95.pattern == "4.4.4.4") & (cells95.J == J)]
    c = c.iloc[(c.D - Dq).abs().argmin()]        # nearest simulated configuration
    rec.append(c)

stats = pd.DataFrame([dict(
    J=int(c.J), D=round(c.D), R_mean=round(c.R_mean * 100, 2),
    R_median=round(c.R_median * 100, 2), worst=round(c.R_min_design * 100, 2),
    designs=int(c.designs_ok), var_pt=round(c.variant_pt * 100, 1),
    var_lb=round(c.variant_lb * 100, 1), design_lb=round(c.design_lb * 100, 1),
    p=c.p_wilcoxon) for c in rec])
stats["holm_p"] = holm(stats.p.values)
stats["p"] = stats.p.round(5); stats["holm_p"] = stats.holm_p.round(5)
print("RECOMMENDED CONFIGURATIONS (nearest simulated cell to the q=90% frontier)")
print(stats.to_string(index=False))



RECOMMENDED CONFIGURATIONS (nearest simulated cell to the q=90% frontier)
  J   D  R_mean  R_median  worst  designs  var_pt  var_lb  design_lb       p  holm_p
 10  24   97.04     97.03  97.00       17   100.0   100.0       83.8 0.00001 0.00004
 20  50   95.53     95.54  95.37       17    98.2    96.5       83.8 0.00001 0.00004
 40  96   96.18     96.18  95.55       17    96.5    93.5       83.8 0.00001 0.00004
 80 165   97.57     98.42  92.70       16    95.3    92.9       75.0 0.00011 0.00021
160 180   98.93     99.98  91.93       16    96.5    94.1       75.0 0.00025 0.00025


In [45]:
rec = []
for J in [10, 20, 40, 80, 160]:
    c = cells95[(cells95.pattern == "4.4.4.4") & (cells95.J == J) & (cells95.variant_lb >= 0.90)]
    rec.append(c.loc[c.D.idxmax()])

stats = pd.DataFrame([dict(
    J=int(c.J), D=round(c.D), R_mean=round(c.R_mean * 100, 2),
    R_median=round(c.R_median * 100, 2), worst=round(c.R_min_design * 100, 2),
    wire_m=round(c.wire_m, 2), designs=int(c.designs_ok),
    var_pt=round(c.variant_pt * 100, 1), var_lb=round(c.variant_lb * 100, 1),
    design_lb=round(c.design_lb * 100, 1), p=c.p_wilcoxon) for c in rec])
stats["holm_p"] = np.round(holm(stats.p.values), 5)
stats["p"] = stats.p.round(5)
print("RECOMMENDED CONFIGURATIONS (largest simulated cell meeting q=90%, R_min=95%)")
print(stats.to_string(index=False))

print("\ninterpolated frontier vs nearest verified cell:")
for J in [10, 20, 40, 80, 160]:
    ds = critical_spacing(d, "4.4.4.4", J, r_min=0.95)
    sim = sorted(cells95[(cells95.pattern == "4.4.4.4") & (cells95.J == J)].D.unique())
    print("  J=%3d  frontier %3.0f  simulated nearby: %s"
          % (J, D_at_q_boot(ds, 0.90), [int(x) for x in sim if 0.7 * D_at_q_boot(ds, 0.90) < x < 1.5 * D_at_q_boot(ds, 0.90)]))



RECOMMENDED CONFIGURATIONS (largest simulated cell meeting q=90%, R_min=95%)
  J   D  R_mean  R_median  worst  wire_m  designs  var_pt  var_lb  design_lb       p  holm_p
 10  24   97.04     97.03  97.00   99.92       17   100.0   100.0       83.8 0.00001 0.00004
 20  50   95.53     95.54  95.37   47.91       17    98.2    96.5       83.8 0.00001 0.00004
 40  96   96.18     96.18  95.55   24.88       17    96.5    93.5       83.8 0.00001 0.00004
 80 170   97.44     98.04  93.48   13.98       16    97.1    95.3       75.0 0.00005 0.00011
160 190   98.68     99.95  93.70   12.43       16    95.3    92.4       75.0 0.00021 0.00021

interpolated frontier vs nearest verified cell:
  J= 10  frontier  25  simulated nearby: [20, 24, 28]
  J= 20  frontier  50  simulated nearby: [40, 50, 53, 56, 60]
  J= 40  frontier  96  simulated nearby: [80, 90, 96, 100, 125]
  J= 80  frontier 165  simulated nearby: [125, 160, 165, 168, 170, 174, 176, 180, 206, 224, 240]
  J=160  frontier 182  simulated nearby

In [46]:
# --- manufacturability penalty (S1) and C2a coincidence check (N4) ---
print("MANUFACTURABILITY PENALTY (coverage criterion, q=90%, R_min=95%)")
for J in [10, 20, 40, 80, 160]:
    s = cells95[(cells95.J == J) & (cells95.variant_lb >= 0.90)]
    if not len(s):
        continue
    best_all = s.loc[s.EP.idxmax()]
    m = s[s.pattern.isin(FINALISTS)]
    best_man = m.loc[m.EP.idxmax()]
    pen = (best_all.EP / best_man.EP - 1) * 100
    print("  J=%3d  all 13: %-12s A=%3.0f EP=%5.0f | manufacturable: %-12s D=%3.0f EP=%5.0f | penalty %5.2f%%"
          % (J, best_all.pattern, best_all.A, best_all.EP,
             best_man.pattern, best_man.D, best_man.EP, pen))

print("\nC2a - argmax(E_P) vs argmin(L_W), both subject to the same constraint")
for J in [10, 20, 40, 80, 160]:
    s = cells95[(cells95.J == J) & (cells95.variant_lb >= 0.90) & (cells95.pattern.isin(FINALISTS))]
    if not len(s):
        continue
    a = s.loc[s.EP.idxmax()]
    b = s.loc[s.wire_m.idxmin()]
    same = (a.pattern == b.pattern) and (a.A == b.A)
    print("  J=%3d  max E_P: %-12s D=%3.0f | min wire: %-12s D=%3.0f | %s"
          % (J, a.pattern, a.D, b.pattern, b.D, "same" if same else "DIFFER"))



MANUFACTURABILITY PENALTY (coverage criterion, q=90%, R_min=95%)
  J= 10  all 13: 4.4.4.4      A= 24 EP= 1325 | manufacturable: 4.4.4.4      D= 24 EP= 1325 | penalty  0.00%
  J= 20  all 13: 4.4.4.4      A= 50 EP= 2722 | manufacturable: 4.4.4.4      D= 50 EP= 2722 | penalty  0.00%
  J= 40  all 13: 4.4.4.4      A= 96 EP= 5280 | manufacturable: 4.4.4.4      D= 96 EP= 5280 | penalty  0.00%
  J= 80  all 13: 4.4.4.4      A=170 EP= 9435 | manufacturable: 4.4.4.4      D=170 EP= 9435 | penalty  0.00%
  J=160  all 13: 6.6.6        A=125 EP=12195 | manufacturable: 4.4.4.4      D=190 EP=10870 | penalty 12.19%

C2a - argmax(E_P) vs argmin(L_W), both subject to the same constraint
  J= 10  max E_P: 4.4.4.4      D= 24 | min wire: 4.4.4.4      D= 24 | same
  J= 20  max E_P: 4.4.4.4      D= 50 | min wire: 4.4.4.4      D= 50 | same
  J= 40  max E_P: 4.4.4.4      D= 96 | min wire: 4.4.4.4      D= 96 | same
  J= 80  max E_P: 4.4.4.4      D=170 | min wire: 4.4.4.4      D=170 | same
  J=160  max E_P: 4.4.4.

In [47]:
c = cells95[(cells95.pattern == "6.6.6") & (cells95.A == 125) & (cells95.J == 160)].iloc[0]
print("6.6.6 A=125 J=160: R_mean %.2f  worst %.2f  designs %d/17  var_pt %.1f  var_lb %.1f  wire %.2f m"
      % (c.R_mean*100, c.R_min_design*100, c.designs_ok, c.variant_pt*100, c.variant_lb*100, c.wire_m))
print("4.4.4.4 A=190 J=160 wire: %.2f m"
      % cells95[(cells95.pattern=="4.4.4.4")&(cells95.A==190)&(cells95.J==160)].wire_m.iloc[0])



6.6.6 A=125 J=160: R_mean 98.31  worst 93.96  designs 16/17  var_pt 94.1  var_lb 91.8  wire 10.96 m
4.4.4.4 A=190 J=160 wire: 12.43 m


In [48]:
rows = []
for J in [10, 20, 40, 80, 160]:
    sub = cells95[(cells95.J == J) & (cells95.variant_lb >= 0.90)]
    if not len(sub):
        continue
    best = sub.loc[sub.groupby("pattern").wire_m.idxmin()]
    for _, c in best.iterrows():
        g = d[(d.pattern == c.pattern) & (d.A == c.A) & (d.J == J)]
        rows.append(dict(J=int(J), pattern=c.pattern, deg=deg[c.pattern], A=int(c.A),
                         wire_m=round(c.wire_m, 2),
                         geom=round(g.unreach_geom.mean() * 100, 2),
                         topo=round((1 - g.reach - g.unreach_geom).mean() * 100, 2)))
t = pd.DataFrame(rows).sort_values(["J", "wire_m"])
print(t.to_string(index=False))



  J     pattern  deg   A  wire_m  geom  topo
 10     4.4.4.4    4  24   99.92  2.96  0.00
 10 3.3.3.3.3.3    6  40  103.78  2.06 -0.00
 10     3.6.3.6    4  13  156.21  1.00 -0.00
 10   3.3.3.4.4    5  20  160.64  0.01  0.00
 10   3.3.4.3.4    5  20  160.65  0.01  0.00
 20     4.4.4.4    4  50   47.91  4.47 -0.00
 20 3.3.3.3.3.3    6  84   49.35  3.78  0.00
 20       4.8.8    3  20   61.70  2.62  0.00
 20    4.6.12.b    3  20   62.67  2.34  0.00
 20       6.6.6    3  20   69.22  0.01 -0.00
 20    4.6.12.a    3  20   73.61  0.01  0.00
 20     3.6.3.6    4  28   74.14  2.50  0.00
 20   3.3.4.3.4    5  40   80.30  0.04 -0.00
 20   3.3.3.4.4    5  40   80.31  0.04  0.00
 20     3.4.6.4    4  20  111.30  0.00 -0.00
 20   3.3.3.3.6    5  20  148.32  0.00 -0.00
 40     4.4.4.4    4  96   24.88  3.82  0.00
 40       6.6.6    3  55   25.05  3.34  0.01
 40    4.6.12.a    3  50   29.36  0.69  0.03
 40       4.8.8    3  40   30.82  2.91 -0.00
 40    4.6.12.b    3  40   31.33  2.45 -0.00
 40 3.3.3.

In [49]:
for p, A in [("3.3.3.3.3.3", 100), ("4.4.4.4", 100), ("6.6.6", 100), ("3.6.3.6", 100)]:
    g = d[(d.pattern == p) & (d.A == A)]
    if len(g):
        area = 1.3  # m2, approximate garment area; ratios are what matter
        print("%-12s A=%d  wire %.2f m  ratio to square %.3f"
              % (p, A, g.wire_m.mean(), g.wire_m.mean() / d[(d.pattern=="4.4.4.4")&(d.A==A)].wire_m.mean()))


3.3.3.3.3.3  A=100  wire 41.41 m  ratio to square 1.734
4.4.4.4      A=100  wire 23.89 m  ratio to square 1.000
6.6.6        A=100  wire 13.78 m  ratio to square 0.577
3.6.3.6      A=100  wire 20.69 m  ratio to square 0.866


In [50]:
plane_all = {"3.3.3.3.3.3": 4.4617*0.8660254, "4.4.4.4": 2.5760,
             "3.6.3.6": 1.5560*1.7320508, "6.6.6": 2.5760/0.5773503*0.5773503}
# for 6.6.6 the plane frontier in D/J is the same as the square's, since wire*r=1 for both
for p in ["3.3.3.3.3.3", "4.4.4.4", "6.6.6", "3.6.3.6"]:
    ds = critical_spacing(d, p, 160, r_min=0.95) if p in SPACING else None
    print(p, "needs SPACING entry" if ds is None else "")



3.3.3.3.3.3 
4.4.4.4 
6.6.6 needs SPACING entry
3.6.3.6 


In [51]:
INRADIUS = {"3.3.3.3.3.3": 1/(2*np.sqrt(3)), "4.4.4.4": 0.5,
            "6.6.6": np.sqrt(3)/2, "3.6.3.6": np.sqrt(3)/2}

def critical_r(d, pattern, J, r_min=0.95):
    s = d[(d.pattern == pattern) & (d.J == J)]
    out = {}
    for (cid, size), g in s.groupby(["clothing_id", "size"]):
        g = g.groupby("A").reach.mean().sort_index()
        if len(g) < 3:
            continue
        R = g.index.values * INRADIUS[pattern]
        v = g.values
        if v[0] < r_min:
            out[(cid, size)] = 0.0; continue
        if v[-1] >= r_min:
            out[(cid, size)] = np.inf; continue
        f = PchipInterpolator(R, v - r_min, extrapolate=False)
        grid = np.linspace(R[0], R[-1], 4000)
        w = f(grid)
        cross = np.where(np.diff(np.sign(w)) < 0)[0]
        out[(cid, size)] = grid[cross[0]] if len(cross) else R[-1]
    return pd.Series(out)

# plane prediction: uncovered = ((r-J)/r)^2 = 0.05  ->  r = J/(1-sqrt(0.05)) = 1.288 J
for p in ["6.6.6", "4.4.4.4", "3.3.3.3.3.3", "3.6.3.6"]:
    rs = critical_r(d, p, 160)
    if not len(rs):
        continue
    lo = D_at_q_boot(rs, 0.90, hi=600)
    print("%-12s  r = %5.1f mm   r/J = %.2f   plane 1.29   frac %.2f"
          % (p, lo, lo/160, lo/160/1.288))



6.6.6         r =  92.7 mm   r/J = 0.58   plane 1.29   frac 0.45
4.4.4.4       r =  91.2 mm   r/J = 0.57   plane 1.29   frac 0.44
3.3.3.3.3.3   r =  77.1 mm   r/J = 0.48   plane 1.29   frac 0.37
3.6.3.6       r = 115.5 mm   r/J = 0.72   plane 1.29   frac 0.56


In [52]:
p, J = "6.6.6", 160
s = d[(d.pattern == p) & (d.J == J)]
print("simulated A:", sorted(s.A.unique()))
print("\nmean reachability by A:")
print((s.groupby("A").reach.mean() * 100).round(2).to_string())
print("\ncell-level coverage bounds:")
print(cells95[(cells95.pattern == p) & (cells95.J == J)][["A", "D", "R_mean", "variant_lb"]].to_string(index=False))

rs = critical_r(d, p, J)
print("\ncritical r: min %.1f  median %.1f  max %s  n=%d"
      % (rs.min(), rs.median(), "inf" if np.isinf(rs.max()) else "%.1f" % rs.max(), len(rs)))
print("variants with r* below 108 (i.e. claiming A=125 fails):", int((rs < 108.3).sum()))



simulated A: [np.float64(20.0), np.float64(40.0), np.float64(80.0), np.float64(100.0), np.float64(125.0), np.float64(135.0), np.float64(145.0), np.float64(160.0), np.float64(206.0)]

mean reachability by A:
A
20.0     100.00
40.0     100.00
80.0      99.79
100.0     99.43
125.0     98.31
135.0     96.80
145.0     94.50
160.0     93.03
206.0     82.32

cell-level coverage bounds:
    A   D   R_mean  variant_lb
 20.0 NaN 1.000000    1.000000
 40.0 NaN 1.000000    1.000000
 80.0 NaN 0.997903    0.982353
100.0 NaN 0.994274    0.941176
125.0 NaN 0.983096    0.917647
135.0 NaN 0.968037    0.847059
145.0 NaN 0.945004    0.782353
160.0 NaN 0.930281    0.723529
206.0 NaN 0.823202    0.341176

critical r: min 43.6  median 167.2  max inf  n=170
variants with r* below 108 (i.e. claiming A=125 fails): 13


In [53]:
# wire consumed per unit coverage, measured at a common cell (J=80, A=100 from tier 4)
s = d[(d.J == 80) & (d.A == 100)].groupby("pattern").agg(R=("reach", "mean"), W=("wire_m", "mean"))
s["wire_per_cov"] = s.W / s.R
print(s.sort_values("wire_per_cov").round(3).to_string())


               R       W  wire_per_cov
pattern                               
6.6.6      0.968  13.755        14.216
4.8.8      0.857  12.222        14.269
4.6.12.b   0.858  12.402        14.459
4.6.12.a   0.970  14.505        14.951
4.6.12     0.710  10.952        15.420
3.12.12    0.516   8.746        16.959
3.4.6.4    0.993  22.170        22.337
3.3.3.3.6  0.990  29.488        29.774
3.3.4.3.4  1.000  31.991        31.999
3.3.3.4.4  1.000  31.992        32.005


In [54]:
print("Best configuration of each pattern (coverage bound >= 90%, R_min 95%), ranked by wire")
for J in [10, 20, 40, 80, 160]:
    s = cells95[(cells95.J == J) & (cells95.variant_lb >= 0.90)]
    if not len(s):
        continue
    best = s.loc[s.groupby("pattern").wire_m.idxmin()].sort_values("wire_m")
    print(" J=%3d" % J)
    for _, r in best.head(5).iterrows():
        print("    %-12s A=%5.0f  wire %6.2f m  lb %5.1f%% %s"
              % (r.pattern, r.A, r.wire_m, r.variant_lb * 100,
                 "*" if r.pattern in FINALISTS else ""))
    m = best[best.pattern.isin(FINALISTS)]
    if len(m) >= 2:
        print("    square over triangle: %+.1f%%" % ((m.iloc[1].wire_m / m.iloc[0].wire_m - 1) * 100))
    if len(best) and best.iloc[0].pattern not in FINALISTS:
        print("    MANUFACTURABILITY PENALTY: %.1f%% (%s beats %s)"
              % ((m.iloc[0].wire_m / best.iloc[0].wire_m - 1) * 100,
                 best.iloc[0].pattern, m.iloc[0].pattern))
    else:
        print("    manufacturability penalty: 0.0%")



Best configuration of each pattern (coverage bound >= 90%, R_min 95%), ranked by wire
 J= 10
    4.4.4.4      A=   24  wire  99.92 m  lb 100.0% *
    3.3.3.3.3.3  A=   40  wire 103.78 m  lb 100.0% *
    3.6.3.6      A=   13  wire 156.21 m  lb 100.0% *
    3.3.3.4.4    A=   20  wire 160.64 m  lb 100.0% 
    3.3.4.3.4    A=   20  wire 160.65 m  lb 100.0% 
    square over triangle: +3.9%
    manufacturability penalty: 0.0%
 J= 20
    4.4.4.4      A=   50  wire  47.91 m  lb  96.5% *
    3.3.3.3.3.3  A=   84  wire  49.35 m  lb  98.2% *
    4.8.8        A=   20  wire  61.70 m  lb 100.0% 
    4.6.12.b     A=   20  wire  62.67 m  lb  98.2% 
    6.6.6        A=   20  wire  69.22 m  lb 100.0% 
    square over triangle: +3.0%
    manufacturability penalty: 0.0%
 J= 40
    4.4.4.4      A=   96  wire  24.88 m  lb  93.5% *
    6.6.6        A=   55  wire  25.05 m  lb  94.1% 
    4.6.12.a     A=   50  wire  29.36 m  lb  95.9% 
    4.8.8        A=   40  wire  30.82 m  lb  98.2% 
    4.6.12.b     A=   4